# Trade Idea #01 — Long Dispersion on the S&P 500 (short implied correlation)

**Author:** Jonathan Phung · ESCP MiM (Market Finance)

This notebook builds every number used in the trade pitch (`README.md`). Nothing is typed in by hand:
all levels come from free data downloaded at run time.

| Step | What it does | Data source |
|---|---|---|
| 1 | Top-N S&P 500 constituents and weights | SSGA SPY daily holdings (fallback: Wikipedia list + Yahoo market caps) |
| 2 | **Snapshot** ATM implied vols (1M, 3M) for SPX and each component, then **implied correlation** | Yahoo option chains via `yfinance` (today only) |
| 3 | **Realized correlation** history (21d and 63d windows) with the same formula | Yahoo daily prices |
| 4 | Cboe implied correlation indices **COR1M / COR3M**, VIX, VIX3M | Yahoo (`^COR1M`, `^COR3M`, `^VIX`, `^VIX3M`) |
| 5 | **Backtest**: short 3M correlation, struck at COR3M, settled on realized correlation (daily ladder, approximate mark-to-market) | from 3 and 4 |
| 6 | Charts + `levels.json` + README auto-fill | — |

**What free data cannot give us (and what we do instead)**
* *Historical* single-stock implied vols are not free (OptionMetrics / Cboe DataShop / Bloomberg are paid).
  Yahoo only serves **today's** option chains. So our own implied-correlation estimate is a **snapshot**,
  and the **history** of implied correlation comes from Cboe's COR1M / COR3M indices.
* Historical index weights are not free either. We apply **today's** top-N weights to the whole history:
  this creates look-ahead and survivorship bias in realized correlation (discussed in the limitations section).
* Yahoo option data can be stale or have zero bid/ask outside US market hours. **Run this notebook during US
  trading hours (15:30–22:00 Paris time)** for clean quotes. The code flags names priced on last trades.

In [ ]:
# --- Setup (Colab): make sure a recent yfinance is installed --------------------------------------
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
# Install/upgrade BEFORE the first import (upgrading an already-imported module has no effect).
# If you already imported yfinance in this runtime, use Runtime > Restart session, then run all.
from importlib import metadata
try:
    print("yfinance installed:", metadata.version("yfinance"))
except metadata.PackageNotFoundError:
    print("yfinance not installed")
if IN_COLAB or "yfinance" not in {d.metadata["Name"].lower() for d in metadata.distributions()}:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "yfinance"], check=False)

In [ ]:
import os, io, json, time, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
import yfinance as yf
from scipy.optimize import brentq
from scipy.stats import norm
try:
    from IPython.display import display
except ImportError:  # plain Python
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 160)

CFG = {
    "N_COMPONENTS": 50,                     # Cboe COR3M also uses the top 50 SPX names
    "INDEX_OPTION_TICKERS": ["^SPX", "SPY"],# SPX (European) first, SPY (American) as fallback
    "INDEX_PRICE_TICKER": "^GSPC",          # S&P 500 price index for realized vol
    "TENORS_DAYS": {"1M": 30, "3M": 91},    # constant-maturity targets, calendar days
    "MIN_EXPIRY_DAYS": 7,                   # ignore expiries closer than this (pin / noise)
    "MONEYNESS_BAND": 0.10,                 # use strikes within +/-10% of spot
    "MAX_REL_SPREAD": 0.50,                 # reject quotes with (ask-bid)/mid above this
    "HIST_START": "2006-01-01",
    "WINDOWS": {"1M": 21, "3M": 63},        # realized-correlation windows, trading days
    "BACKTEST_TENOR": 63,                   # 3M correlation swap proxy
    "STRIKE_HAIRCUTS": [0, 3, 5, 8],        # COR3M is not executable: test strikes below it (corr. points)
    "TRADE_N": 25,                          # names actually traded in the proposed book
    "STRESS_VIX": 30.0,                     # shade days with VIX close above this
    "MIN_WEIGHT_COVERAGE": 0.80,            # min share of basket weight with data on a given date
    "OUT": "output",
    # Optional: raw URL of README_template.md in your repo, used to auto-fill the pitch
    "README_TEMPLATE_URL": "https://raw.githubusercontent.com/Jojolap1/quant-finance-projects/main/trade-ideas/01-dispersion/README_template.md",
    # Optional manual universe if both holdings sources fail, e.g. ["AAPL","MSFT",...]
    "MANUAL_TICKERS": None,
}
FIG_DIR = os.path.join(CFG["OUT"], "figures")
os.makedirs(FIG_DIR, exist_ok=True)
UA = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) research-notebook"}

# --- Chart style: one consistent, recessive look for README images --------------------------------
C = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "red": "#e34948",
     "ink": "#0b0b0b", "ink2": "#52514e", "grid": "#e4e3df", "shade": "#ecebe7", "surface": "#fcfcfb"}
plt.rcParams.update({
    "figure.facecolor": C["surface"], "axes.facecolor": C["surface"], "savefig.facecolor": C["surface"],
    "axes.edgecolor": C["ink2"], "axes.labelcolor": C["ink2"], "xtick.color": C["ink2"], "ytick.color": C["ink2"],
    "axes.grid": True, "grid.color": C["grid"], "grid.linewidth": 0.8, "axes.spines.top": False,
    "axes.spines.right": False, "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlecolor": C["ink"], "legend.frameon": False, "lines.linewidth": 1.6, "figure.dpi": 110,
})

def save_fig(fig, name):
    path = os.path.join(FIG_DIR, name)
    fig.savefig(path, dpi=160, bbox_inches="tight")
    print("saved", path)
    plt.show()

## 0. Data helpers

In [ ]:
def fetch_close(tickers, start, retries=3):
    """Download adjusted daily closes for a list of tickers. Returns a DataFrame (dates x tickers).
    Tickers that fail come back as all-NaN columns (and are reported), never as invented data."""
    tickers = list(dict.fromkeys(tickers))
    df = None
    for k in range(retries):
        try:
            df = yf.download(tickers, start=start, auto_adjust=True, progress=False,
                             threads=True, group_by="column")
            if df is not None and not df.empty:
                break
        except Exception as e:
            print(f"  download attempt {k+1} failed: {e}")
        time.sleep(2 * (k + 1))
    if df is None or df.empty:
        return pd.DataFrame(columns=tickers, dtype=float)
    close = df["Close"] if isinstance(df.columns, pd.MultiIndex) else df[["Close"]].rename(columns={"Close": tickers[0]})
    if isinstance(close, pd.Series):
        close = close.to_frame(tickers[0])
    close.index = pd.to_datetime(close.index)
    if close.index.tz is not None:
        close.index = close.index.tz_localize(None)
    close = close.reindex(columns=tickers)
    missing = [t for t in tickers if close[t].notna().sum() == 0]
    if missing:
        print(f"  no data for: {missing}")
    return close.dropna(how="all")

def to_yahoo(t):
    """Holdings files use 'BRK.B'; Yahoo uses 'BRK-B'."""
    return str(t).strip().upper().replace(".", "-").replace("/", "-")

# Dual share classes are economically one company: merging avoids an artificial ~1.0 pairwise correlation.
SHARE_CLASS_MERGE = {"GOOG": "GOOGL", "FOX": "FOXA", "NWS": "NWSA"}

## 1. Universe: top-N S&P 500 constituents and weights

Primary source: the SPY daily holdings file published by State Street (SSGA). SPY fully replicates the
S&P 500, so its weights are float-adjusted index weights. Fallback: the S&P 500 list from Wikipedia, weighted
by Yahoo market caps (**not** float-adjusted, so slightly different from true index weights).

As in Cboe's methodology, weights are **re-normalized to sum to 1 inside the top-N basket**.

In [ ]:
SSGA_URL = ("https://www.ssga.com/us/en/intermediary/library-content/products/fund-data/etfs/us/"
            "holdings-daily-us-en-spy.xlsx")

def weights_from_ssga():
    r = requests.get(SSGA_URL, headers=UA, timeout=30)
    r.raise_for_status()
    raw = pd.read_excel(io.BytesIO(r.content), header=None)
    is_hdr = raw.apply(lambda row: any(isinstance(v, str) and v.strip() == "Ticker" for v in row), axis=1)
    hdr = int(raw.index[is_hdr][0])
    as_of = ""
    for v in raw.iloc[:hdr].values.ravel():
        if isinstance(v, str) and "as of" in v.lower():
            as_of = v.strip()
    df = pd.read_excel(io.BytesIO(r.content), header=hdr)
    df.columns = [str(c).strip() for c in df.columns]
    df = df[["Ticker", "Name", "Weight"]].copy()
    df["Weight"] = pd.to_numeric(df["Weight"], errors="coerce")
    df["Ticker"] = df["Ticker"].astype(str).str.strip()
    df = df.dropna(subset=["Weight"])
    df = df[df["Ticker"].str.match(r"^[A-Z][A-Z0-9.\-/]*$")]          # drops cash / futures lines
    return df, f"SSGA SPY daily holdings ({as_of or 'date not parsed'})"

def weights_from_wikipedia_mcap(tickers=None):
    if tickers is None:
        html = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies", headers=UA, timeout=30).text
        tickers = pd.read_html(io.StringIO(html))[0]["Symbol"].astype(str).tolist()
        src = "Wikipedia S&P 500 list + Yahoo market caps (not float-adjusted)"
    else:
        src = "Manual ticker list + Yahoo market caps (not float-adjusted)"
    rows = []
    for t in tickers:
        try:
            mc = yf.Ticker(to_yahoo(t)).fast_info["market_cap"]
            if mc and np.isfinite(mc) and mc > 0:
                rows.append((t, t, float(mc)))
        except Exception:      # includes Yahoo rate limits: the name is skipped, never guessed
            pass
        time.sleep(0.1)
    df = pd.DataFrame(rows, columns=["Ticker", "Name", "Weight"])
    df["Weight"] = 100 * df["Weight"] / df["Weight"].sum()
    return df, src

holdings, WEIGHT_SOURCE = None, None
for fn in (weights_from_ssga, weights_from_wikipedia_mcap):
    try:
        holdings, WEIGHT_SOURCE = fn()
        if len(holdings) >= CFG["N_COMPONENTS"]:
            break
    except Exception as e:
        print(f"{fn.__name__} failed: {e}")
if (holdings is None or len(holdings) < CFG["N_COMPONENTS"]) and CFG["MANUAL_TICKERS"]:
    holdings, WEIGHT_SOURCE = weights_from_wikipedia_mcap(CFG["MANUAL_TICKERS"])
if holdings is None or holdings.empty:
    raise RuntimeError("No constituent data. Set CFG['MANUAL_TICKERS'] to a list of tickers and re-run.")

holdings["Ticker"] = holdings["Ticker"].map(to_yahoo).replace(SHARE_CLASS_MERGE)
holdings = (holdings.groupby("Ticker", as_index=False)
                    .agg(Name=("Name", "first"), Weight=("Weight", "sum"))
                    .sort_values("Weight", ascending=False))
top = holdings.head(CFG["N_COMPONENTS"]).copy()
INDEX_COVERAGE = top["Weight"].sum() / holdings["Weight"].sum()
top["w"] = top["Weight"] / top["Weight"].sum()          # basket weights, sum to 1
W = top.set_index("Ticker")["w"]
print(f"Weights source: {WEIGHT_SOURCE}")
print(f"Top {len(top)} names = {INDEX_COVERAGE:.1%} of index weight")
top.head(10)

## 2. Snapshot: ATM implied volatility and implied correlation

**Implied vol extraction (per underlying, per expiry)** — built to be robust to Yahoo data quality instead of
trusting Yahoo's own `impliedVolatility` field (kept only as a cross-check):
1. Mid prices from bid/ask (quotes with zero bid, crossed or > 50% wide are rejected).
2. **Forward and discount factor from put–call parity**: for strikes within ±10% of spot,
   $C - P = DF\,(F - K)$, so a regression of $C-P$ on $K$ gives slope $-DF$ and intercept $DF\cdot F$.
   This absorbs rates, dividends and borrow without needing any input we would have to assume.
3. Black-76 implied vols on **out-of-the-money** options (calls above F, puts below F).
4. **ATM-forward vol** = local quadratic fit of IV in log-moneyness $k=\ln(K/F)$, evaluated at $k=0$.
5. **Constant maturity**: linear interpolation in total variance $\sigma^2T$ between the two expiries bracketing
   30d / 91d.

Caveat: single-stock options are American. Near ATM and at short maturities the early-exercise premium is small,
but it slightly biases parity-implied forwards; SPX options are European and unaffected.

**Implied correlation.** With basket weights $w_i$, index vol $\sigma_I$ and component vols $\sigma_i$:

$$\sigma_I^2=\sum_i w_i^2\sigma_i^2+\sum_{i\neq j} w_iw_j\rho_{ij}\sigma_i\sigma_j .$$

Replacing every $\rho_{ij}$ by a single $\rho$ and solving (this is the Cboe COR formula):

$$\rho_{impl}=\frac{\sigma_I^2-\sum_i w_i^2\sigma_i^2}{\big(\sum_i w_i\sigma_i\big)^2-\sum_i w_i^2\sigma_i^2}
\;\approx\;\left(\frac{\sigma_I}{\sum_i w_i\sigma_i}\right)^2 \quad(\text{large }N).$$

Approximations to keep in mind: (i) the SPX has ~500 names but the basket has N; the index vol is used as if it
were the basket vol (same convention as Cboe); (ii) ATM vols, not variance-swap levels, so skew is ignored;
(iii) a single average $\rho$ summarizes the whole correlation matrix.

In [ ]:
def black76(F, K, T, sig, DF, is_call):
    sd = sig * np.sqrt(T)
    d1 = (np.log(F / K) + 0.5 * sd * sd) / sd
    d2 = d1 - sd
    if is_call:
        return DF * (F * norm.cdf(d1) - K * norm.cdf(d2))
    return DF * (K * norm.cdf(-d2) - F * norm.cdf(-d1))

def implied_vol(price, F, K, T, DF, is_call):
    intrinsic = DF * max(F - K, 0.0) if is_call else DF * max(K - F, 0.0)
    upper = DF * F if is_call else DF * K
    if not (np.isfinite(price) and intrinsic + 1e-8 < price < upper):
        return np.nan
    try:
        return brentq(lambda s: black76(F, K, T, s, DF, is_call) - price, 1e-4, 5.0, maxiter=200)
    except ValueError:
        return np.nan

def _clean_quotes(df):
    bid, ask = df["bid"].astype(float), df["ask"].astype(float)
    mid = (bid + ask) / 2
    ok = (bid > 0) & (ask > 0) & (ask >= bid) & ((ask - bid) / mid <= CFG["MAX_REL_SPREAD"])
    return mid.where(ok), ok

_chain_cache = {}
def get_chain(tk_symbol, expiry):
    key = (tk_symbol, expiry)
    if key not in _chain_cache:
        ch = yf.Ticker(tk_symbol).option_chain(expiry)
        _chain_cache[key] = (ch.calls.copy(), ch.puts.copy())
    return _chain_cache[key]

def atm_vol_one_expiry(symbol, expiry, spot, T):
    calls, puts = get_chain(symbol, expiry)
    cols = ["strike", "bid", "ask", "lastPrice", "impliedVolatility"]
    if symbol == "^SPX" and "contractSymbol" in calls:
        # Yahoo may mix AM-settled SPX and PM-settled SPXW on the same date: keep SPXW only
        wc, wp = calls["contractSymbol"].str.startswith("SPXW"), puts["contractSymbol"].str.startswith("SPXW")
        if wc.any() and wp.any():
            calls, puts = calls[wc], puts[wp]
    # any remaining duplicate strikes: keep the most open contract
    calls = calls.sort_values("openInterest", ascending=False).drop_duplicates("strike") if "openInterest" in calls else calls.drop_duplicates("strike")
    puts = puts.sort_values("openInterest", ascending=False).drop_duplicates("strike") if "openInterest" in puts else puts.drop_duplicates("strike")
    c, p = calls[cols].copy(), puts[cols].copy()
    c["mid"], c["ok"] = _clean_quotes(c)
    p["mid"], p["ok"] = _clean_quotes(p)
    m = c.merge(p, on="strike", suffixes=("_c", "_p"))
    band = m[(m["strike"] / spot - 1).abs() <= CFG["MONEYNESS_BAND"]].copy()
    price_src = "mid"
    pair = band[band["ok_c"] & band["ok_p"]]
    if len(pair) < 3:   # e.g. market closed and Yahoo shows zero bid/ask -> fall back to last trades (stale!)
        price_src = "last"
        band["mid_c"], band["mid_p"] = band["lastPrice_c"], band["lastPrice_p"]
        pair = band[(band["mid_c"] > 0) & (band["mid_p"] > 0)]
    if len(pair) < 3:
        raise ValueError("fewer than 3 usable strikes")
    # Put-call parity regression  C - P = DF*F - DF*K
    x, y = pair["strike"].values.astype(float), (pair["mid_c"] - pair["mid_p"]).values.astype(float)
    slope, icpt = np.polyfit(x, y, 1)
    resid = y - (slope * x + icpt)
    tol = max(4 * 1.4826 * np.median(np.abs(resid - np.median(resid))), 1e-4 * spot)
    keep = np.abs(resid) <= tol
    if keep.sum() >= 3:
        slope, icpt = np.polyfit(x[keep], y[keep], 1)
    DF = -slope
    if 0.85 < DF <= 1.02:
        F = icpt / DF
    else:                 # implausible fit: DF=1 and forward from the strike where C-P is smallest
        DF = 1.0
        i = int(np.argmin(np.abs(y)))
        F = x[i] + y[i]
    # OTM implied vols
    rows = []
    for _, r in band.iterrows():
        K = float(r["strike"])
        is_call = K >= F
        price = r["mid_c"] if is_call else r["mid_p"]
        rows.append((np.log(K / F), implied_vol(price, F, K, T, DF, is_call),
                     r["impliedVolatility_c"] if is_call else r["impliedVolatility_p"]))
    s = pd.DataFrame(rows, columns=["k", "iv", "yahoo_iv"]).dropna(subset=["iv"])
    if len(s) < 2 or not ((s["k"] <= 0).any() and (s["k"] >= 0).any()):
        raise ValueError("ATM not bracketed by valid strikes")
    near = s.assign(a=s["k"].abs()).nsmallest(6, "a").sort_values("k")
    deg = 2 if len(near) >= 5 else 1
    atm = float(np.polyval(np.polyfit(near["k"], near["iv"], deg), 0.0))
    if not (0.8 * near["iv"].min() <= atm <= 1.2 * near["iv"].max()):   # guard against a bad fit
        atm = float(np.interp(0.0, near["k"], near["iv"]))
    yahoo_atm = float(np.interp(0.0, near["k"], near["yahoo_iv"])) if near["yahoo_iv"].notna().all() else np.nan
    return {"expiry": expiry, "T": T, "F": F, "DF": DF, "atm_vol": atm, "n_strikes": len(s),
            "price_src": price_src, "yahoo_atm_iv": yahoo_atm}

def spot_of(symbol):
    tk = yf.Ticker(symbol)
    try:
        px = float(tk.fast_info["last_price"])
        if np.isfinite(px) and px > 0:
            return px
    except Exception:
        pass
    return float(tk.history(period="5d")["Close"].dropna().iloc[-1])

def atm_vol_tenors(symbol, tenors_days, val_date):
    """Constant-maturity ATM vols for several tenors. Returns {tenor: dict}."""
    tk = yf.Ticker(symbol)
    expiries = list(tk.options)
    if not expiries:
        raise ValueError("no listed expiries on Yahoo")
    spot = spot_of(symbol)
    days = {e: (pd.Timestamp(e) - val_date).days for e in expiries}
    usable = sorted([e for e in expiries if days[e] >= CFG["MIN_EXPIRY_DAYS"]], key=lambda e: days[e])
    out = {}
    for name, target in tenors_days.items():
        lo = [e for e in usable if days[e] <= target]
        hi = [e for e in usable if days[e] > target]
        picks = ([lo[-1]] if lo else []) + ([hi[0]] if hi else [])
        res = []
        for e in picks:
            try:
                res.append(atm_vol_one_expiry(symbol, e, spot, days[e] / 365.0))
            except Exception as ex:
                print(f"    {symbol} {e}: {ex}")
        if not res:
            continue
        Tt = target / 365.0
        if len(res) == 2:
            (r1, r2) = res
            w1, w2 = r1["atm_vol"] ** 2 * r1["T"], r2["atm_vol"] ** 2 * r2["T"]
            wt = w1 + (w2 - w1) * (Tt - r1["T"]) / (r2["T"] - r1["T"])
            vol, flag = np.sqrt(max(wt, 1e-8) / Tt), ("" if w2 >= w1 else "calendar-inverted")
        else:
            vol, flag = res[0]["atm_vol"], f"single expiry ({res[0]['expiry']})"
        out[name] = {"vol": vol, "expiries": "/".join(r["expiry"] for r in res), "flag": flag,
                     "price_src": "/".join(r["price_src"] for r in res),
                     "yahoo_iv": np.nanmean([r["yahoo_atm_iv"] for r in res]), "spot": spot}
    return out

def implied_correlation(sig_idx, w, sig):
    """Cboe-style implied correlation and its large-N proxy. w are renormalized on available names."""
    w = np.asarray(w, float); sig = np.asarray(sig, float)
    w = w / w.sum()
    a = np.sum(w ** 2 * sig ** 2)
    b = np.sum(w * sig) ** 2
    return (sig_idx ** 2 - a) / (b - a), (sig_idx / np.sum(w * sig)) ** 2

In [ ]:
# Valuation date = last S&P 500 close (option quotes on Yahoo refer to the latest session)
_idx_last = fetch_close([CFG["INDEX_PRICE_TICKER"]], start=(pd.Timestamp.today() - pd.Timedelta(days=15)).strftime("%Y-%m-%d"))
VAL_DATE = _idx_last.index[-1].normalize()
print("Valuation date:", VAL_DATE.date())

index_vols, INDEX_OPT_SYMBOL = None, None
for sym in CFG["INDEX_OPTION_TICKERS"]:
    try:
        v = atm_vol_tenors(sym, CFG["TENORS_DAYS"], VAL_DATE)
        if all(t in v for t in CFG["TENORS_DAYS"]):
            index_vols, INDEX_OPT_SYMBOL = v, sym
            break
        print(f"{sym}: incomplete tenors {list(v)}")
    except Exception as e:
        print(f"{sym} failed: {e}")
if index_vols is None:
    raise RuntimeError("Could not build index ATM vols from Yahoo (try again during US market hours).")
print("Index option underlying:", INDEX_OPT_SYMBOL, {k: round(v["vol"], 4) for k, v in index_vols.items()})

comp_rows = []
for i, t in enumerate(W.index):
    try:
        v = atm_vol_tenors(t, CFG["TENORS_DAYS"], VAL_DATE)
        for ten, d in v.items():
            comp_rows.append({"ticker": t, "tenor": ten, "w": W[t], **d})
    except Exception as e:
        print(f"  {t}: skipped ({e})")
    time.sleep(0.15)   # be polite to Yahoo
    if (i + 1) % 10 == 0:
        print(f"  ... {i+1}/{len(W)} names done")
comp = pd.DataFrame(comp_rows)
if comp.empty:
    raise RuntimeError("No component option data from Yahoo (try again during US market hours).")

snap = {}
for ten in CFG["TENORS_DAYS"]:
    d = comp[(comp["tenor"] == ten) & comp["vol"].between(0.03, 3.0)]
    cover = d["w"].sum()
    rho, rho_proxy = implied_correlation(index_vols[ten]["vol"], d["w"], d["vol"])
    snap[ten] = {"index_vol": index_vols[ten]["vol"], "n_names": len(d), "basket_weight_covered": cover,
                 "wavg_comp_vol": float(np.sum(d["w"] / d["w"].sum() * d["vol"])),
                 "implied_corr": float(rho), "implied_corr_proxy": float(rho_proxy),
                 "share_last_price": float(d.loc[d["price_src"].str.contains("last"), "w"].sum() / cover) if cover else np.nan}
snap_df = pd.DataFrame(snap).T
comp.to_csv(os.path.join(CFG["OUT"], "snapshot_component_vols.csv"), index=False)
snap_df

## 3. Historical data: prices, VIX, VIX3M and Cboe implied correlation

In [ ]:
hist_px = fetch_close([CFG["INDEX_PRICE_TICKER"]] + list(W.index), CFG["HIST_START"])
vol_idx = fetch_close(["^VIX", "^VIX3M", "^COR1M", "^COR3M", "^COR6M"], CFG["HIST_START"])

# Fallback for COR3M: a CSV downloaded manually from https://www.cboe.com/us/indices/implied/ and uploaded
# to the Colab session as COR3M_History.csv (any file with a date column and a COR3M/close column).
if ("^COR3M" not in vol_idx) or vol_idx["^COR3M"].notna().sum() == 0:
    if os.path.exists("COR3M_History.csv"):
        f = pd.read_csv("COR3M_History.csv")
        dcol = [c for c in f.columns if "date" in c.lower()][0]
        vcol = [c for c in f.columns if c.upper() in ("COR3M", "CLOSE", "VALUE")][0]
        s = pd.Series(pd.to_numeric(f[vcol], errors="coerce").values, index=pd.to_datetime(f[dcol]))
        vol_idx["^COR3M"] = s.reindex(vol_idx.index)
        print("COR3M loaded from local CSV")
    else:
        print("COR3M not available from Yahoo and no local CSV -> backtest cannot run.")

for c in vol_idx.columns:
    s = vol_idx[c].dropna()
    print(f"{c:7s}: " + (f"{s.index[0].date()} -> {s.index[-1].date()}  ({len(s)} obs)" if len(s) else "not available"))

# If the notebook runs during US hours, Yahoo returns today's partial bar: drop it so every series ends at
# the same complete close (the option snapshot itself stays live).
now_ny = pd.Timestamp.now(tz="America/New_York")
if now_ny.time() < pd.Timestamp("16:30").time():
    today = pd.Timestamp(now_ny.date())
    hist_px = hist_px[hist_px.index < today]
    vol_idx = vol_idx[vol_idx.index < today]
px = hist_px.dropna(subset=[CFG["INDEX_PRICE_TICKER"]])
rets = np.log(px).diff().iloc[1:]
r_idx = rets[CFG["INDEX_PRICE_TICKER"]]
R = rets[list(W.index)]
# Remove obviously bad prints (|daily log return| > 50%): set to NaN rather than guess a value
bad = R.abs() > 0.5
if bad.values.sum():
    print(f"Removed {int(bad.values.sum())} suspicious component returns (>50% in a day)")
R = R.mask(bad)

cboe = (vol_idx.reindex(r_idx.index).ffill(limit=3) / 100.0)   # COR in correlation units, VIX in decimals

## 4. Realized correlation (same formula, realized variances)

Variances are **zero-mean** (variance-swap convention) over 21 and 63 trading days. On each date, weights are
renormalized over names that have a full window of data; dates where less than 80% of basket weight has data
are dropped.

In [ ]:
def realized_corr(r_index, Rc, w, window):
    v_idx = (r_index ** 2).rolling(window, min_periods=window).mean()
    V = (Rc ** 2).rolling(window, min_periods=window).mean()     # NaN if any day is missing in the window
    avail = V.notna()
    Wt = avail.mul(w, axis=1)
    coverage = Wt.sum(axis=1)
    Wt = Wt.div(coverage.replace(0, np.nan), axis=0)
    a = (Wt ** 2 * V).sum(axis=1, min_count=1)
    b = (Wt * np.sqrt(V)).sum(axis=1, min_count=1) ** 2
    rho = (v_idx - a) / (b - a)
    rho[coverage < CFG["MIN_WEIGHT_COVERAGE"]] = np.nan
    return rho.clip(-1, 1), coverage

rc = {}
for name, win in CFG["WINDOWS"].items():
    rc[name], cov_ = realized_corr(r_idx, R, W, win)
rc = pd.DataFrame(rc)
print("Realized corr available from", rc["3M"].first_valid_index().date())

# Cross-check on the latest 63d window: weighted average pairwise correlation of the basket itself
# (does not use the index), to see how close index-based and basket-based measures are.
last = R.iloc[-CFG["WINDOWS"]["3M"]:].dropna(axis=1, how="any")
wl = W[last.columns] / W[last.columns].sum()
S = (last.T @ last) / len(last)                    # zero-mean second-moment matrix
sd = np.sqrt(np.diag(S)); Cm = S / np.outer(sd, sd)
num = (np.outer(wl * sd, wl * sd) * Cm).values; den = np.outer(wl * sd, wl * sd)
mask = ~np.eye(len(wl), dtype=bool)
BASKET_PAIRWISE_3M = float(num[mask].sum() / den[mask].sum())
print(f"Latest 3M realized corr: index-based {rc['3M'].dropna().iloc[-1]:.3f} | basket pairwise {BASKET_PAIRWISE_3M:.3f}")

## 5. Why a "realized-only" dispersion backtest would be meaningless

Selling realized index variance and buying weighted realized component variance, **without implied strikes**,
always makes money: by Cauchy–Schwarz and $\rho_{ij}\le 1$,
$\sigma_I^2\le(\sum w_i\sigma_i)^2\le\sum w_i\sigma_i^2$. The real trade earns
*(implied − realized)* on each leg, so it needs implied strikes. Free implied history exists only for the
**index** (VIX, VIX3M) and for **implied correlation** (COR3M) — not for single stocks.

**Honest proxy:** a short **3M correlation swap** struck at COR3M and settled on realized 63-day correlation
(computed above). A vega-neutral / theta-neutral dispersion book is, to first order, short implied correlation,
so this measures the premium the trade harvests — as an **upper bound**: COR3M is an index built from ATM vols,
not an executable correlation-swap strike (dealers quote below it), and implied-minus-realized correlation also
contains part of the index vol risk premium. We therefore re-run the ladder with strikes 3, 5 and 8 points
below COR3M. It ignores vol-level moves, single-stock
idiosyncratic events, gamma path-dependency and transaction costs (see limitations).

**Ladder:** every day we sell 1/63 of a unit, so in steady state the book holds one unit spread across 63
maturities (no start-date luck). **Mark-to-market** of a live tranche entered at $s$, at date $t$ after $e$ days:

$$E_t[\rho_{final}] \approx \tfrac{e}{63}\,\rho^{real}_{s\to t}+\left(1-\tfrac{e}{63}\right)\text{COR3M}_t,\qquad
\text{MtM}=\tfrac{1}{63}\big(K_s-E_t[\rho_{final}]\big)$$

so a spike in COR3M shows up as an immediate loss (this matters for March 2020), not only at expiry.
P&L is in **correlation points per unit** (1 point = 0.01 of correlation).

In [ ]:
H = CFG["BACKTEST_TENOR"]
K = cboe["^COR3M"] if "^COR3M" in cboe else pd.Series(np.nan, index=r_idx.index)
BACKTEST_OK = K.notna().sum() > H + 20
if BACKTEST_OK:
    rho_e = {e: realized_corr(r_idx, R, W, e)[0] for e in range(1, H + 1)}
    # Start once realized correlation is continuously available (some of today's top names have short
    # histories; before that date basket coverage flickers around the threshold).
    rh = rho_e[H]
    gaps = rh.index[rh.isna() & (rh.index > rh.first_valid_index())] if rh.notna().any() else rh.index[:0]
    gaps = gaps[gaps < rh.index[-H]]                       # ignore a missing print in the last weeks
    start_cov = rh.first_valid_index() if len(gaps) == 0 else rh.index[rh.index.get_loc(gaps[-1]) + 1]
    entry_start = max(K.first_valid_index(), start_cov - pd.tseries.offsets.BDay(H))
    K_bt = K.where(K.index >= entry_start)

    def run_ladder(haircut_pts=0.0):
        """Short-correlation ladder struck at COR3M - haircut. Returns (MtM P&L, booked P&L, per-tranche P&L) in pts."""
        Kh = K_bt - haircut_pts / 100.0
        live = pd.Series(0.0, index=K.index)
        for e in range(1, H):                                  # tranches entered e days ago, still alive
            exp_final = (e / H) * rho_e[e].fillna(K) + (1 - e / H) * K
            live = live + ((Kh.shift(e) - exp_final) / H).fillna(0.0)
        locked_daily = (Kh.shift(H) - rho_e[H]) / H            # tranches maturing at t
        locked_cum = locked_daily.fillna(0.0).cumsum()
        st = Kh.first_valid_index()
        mtm = ((locked_cum + live).loc[st:] * 100)[K.loc[st:].notna()]
        return mtm, locked_cum.loc[st:] * 100, ((Kh.shift(H) - rho_e[H]) * 100)

    pnl_mtm, pnl_hold, tranche = run_ladder(0.0)
    n_dropped = int((K_bt.shift(H).notna() & rho_e[H].isna()).sum())   # matured tranches with no settlement
    matured = tranche.dropna()
    daily = pnl_mtm.diff().dropna()
    dd = pnl_mtm - pnl_mtm.cummax()
    trough = dd.idxmin(); peak = pnl_mtm.loc[:trough].idxmax()
    BT = {
        "start": str(pnl_mtm.index[0].date()), "end": str(pnl_mtm.index[-1].date()),
        "years": len(daily) / 252,
        "total_pts": float(pnl_mtm.iloc[-1]),
        "ann_pts": float(daily.mean() * 252), "ann_vol_pts": float(daily.std() * np.sqrt(252)),
        "sharpe": float(daily.mean() / daily.std() * np.sqrt(252)) if daily.std() > 0 else np.nan,
        "max_dd_pts": float(dd.min()), "dd_peak": str(peak.date()), "dd_trough": str(trough.date()),
        "worst_day_pts": float(daily.min()), "worst_day": str(daily.idxmin().date()),
        "hit_rate": float((matured > 0).mean()), "n_tranches": int(len(matured)), "n_dropped": n_dropped,
        "avg_tranche_pts": float(matured.mean()), "p5_tranche_pts": float(matured.quantile(0.05)),
    }
    # Sensitivity to the strike: dealers do not sell correlation at the index level
    hc_rows = []
    for h in CFG["STRIKE_HAIRCUTS"]:
        m_, _, t_ = run_ladder(h)
        d_ = m_.diff().dropna()
        hc_rows.append({"haircut_pts": h, "ann_pts": d_.mean() * 252,
                        "sharpe": d_.mean() / d_.std() * np.sqrt(252) if d_.std() > 0 else np.nan,
                        "max_dd_pts": (m_ - m_.cummax()).min(), "hit_rate": (t_.dropna() > 0).mean()})
    haircut_df = pd.DataFrame(hc_rows)
    display(haircut_df.round(2))
    # Named stress windows (only those inside the sample are reported)
    STRESS = {"GFC (Sep–Nov 2008)": ("2008-09-01", "2008-11-20"), "US downgrade (Jul–Oct 2011)": ("2011-07-22", "2011-10-03"),
              "China deval. (Aug 2015)": ("2015-08-17", "2015-08-25"), "Volmageddon (Feb 2018)": ("2018-01-26", "2018-02-08"),
              "Q4 2018": ("2018-09-20", "2018-12-24"), "COVID (Feb–Mar 2020)": ("2020-02-19", "2020-03-23"),
              "2022 bear market": ("2022-01-03", "2022-10-12"), "Yen carry unwind (Jul–Aug 2024)": ("2024-07-16", "2024-08-05"),
              "Tariff shock (Feb–Apr 2025)": ("2025-02-19", "2025-04-08")}
    stress_rows = []
    for nm, (a_, b_) in STRESS.items():
        seg = pnl_mtm.loc[a_:b_]
        prev = pnl_mtm.loc[:pd.Timestamp(a_) - pd.Timedelta(days=1)]
        if len(seg) > 2 and len(prev) and seg.index[0] <= pd.Timestamp(a_) + pd.Timedelta(days=7):
            stress_rows.append({"window": nm, "from": a_, "to": b_, "pnl_pts": seg.iloc[-1] - prev.iloc[-1],
                                "COR3M_start": K.loc[a_:b_].iloc[0] * 100, "COR3M_max": K.loc[a_:b_].max() * 100,
                                "realized1M_max": rc["1M"].loc[a_:b_].max() * 100})
    stress_df = pd.DataFrame(stress_rows)
    yearly = daily.groupby(daily.index.year).sum().rename("pnl_pts")
    print(json.dumps({k: (round(v, 3) if isinstance(v, float) else v) for k, v in BT.items()}, indent=1))
    display(stress_df.round(2))
else:
    print("Not enough COR3M history for the backtest.")

## 6. Charts (saved to `output/figures/` for the README)

In [ ]:
def shade_stress(ax, vix, thr):
    if vix is None or vix.dropna().empty:
        return
    hi = (vix > thr).astype(int)
    starts = hi.index[(hi.diff() == 1)]
    ends = hi.index[(hi.diff() == -1)]
    if hi.iloc[0] == 1: starts = starts.insert(0, hi.index[0])
    if hi.iloc[-1] == 1: ends = ends.append(pd.Index([hi.index[-1]]))
    for s_, e_ in zip(starts, ends):
        ax.axvspan(s_, e_, color=C["shade"], lw=0, zorder=0)

vix = cboe["^VIX"] * 100 if "^VIX" in cboe else None

# --- Figure 1: implied vs realized correlation ------------------------------------------------------
fig, ax = plt.subplots(figsize=(11, 4.6))
first = K.first_valid_index() if K.notna().any() else rc["3M"].first_valid_index()
view = slice(first - pd.Timedelta(days=30), None)
shade_stress(ax, vix.loc[view] if vix is not None else None, CFG["STRESS_VIX"])
ax.plot(rc["1M"].loc[view] * 100, color=C["aqua"], lw=1.0, label="Realized 1M (21d)")
ax.plot(rc["3M"].loc[view] * 100, color=C["orange"], lw=1.6, label="Realized 3M (63d)")
if K.notna().any():
    ax.plot(K.loc[view] * 100, color=C["blue"], lw=1.8, label="Implied 3M — Cboe COR3M")
for ten, mk in (("3M", "o"), ("1M", "D")):
    ax.scatter([VAL_DATE], [snap[ten]["implied_corr"] * 100], s=55, marker=mk, color=C["ink"], zorder=5,
               edgecolor=C["surface"], linewidth=1.5, label=f"Our snapshot implied {ten} ({VAL_DATE.date()})")
ax.set_ylabel("Correlation (points)")
ax.set_title("S&P 500 top-%d implied vs realized correlation" % len(W), loc="left")
ax.text(0, -0.30, f"Grey bands: VIX close > {CFG['STRESS_VIX']:.0f}. Realized correlation uses today's top-{len(W)} weights. "
        "Sources: Cboe via Yahoo Finance, Yahoo prices, author's calculations.",
        transform=ax.transAxes, fontsize=8, color=C["ink2"])
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3, fontsize=8.5)
save_fig(fig, "01_implied_vs_realized_correlation.png")

# --- Figure 2: correlation risk premium (COR3M at t minus realized over the next 63 days) -----------
if BACKTEST_OK:
    prem = ((K - rc["3M"].shift(-H)) * 100).dropna()
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [3, 1]})
    a1.fill_between(prem.index, 0, prem.where(prem >= 0), color=C["blue"], lw=0, label="Implied > subsequent realized")
    a1.fill_between(prem.index, 0, prem.where(prem < 0), color=C["red"], lw=0, label="Implied < subsequent realized")
    a1.axhline(0, color=C["ink2"], lw=0.8)
    a1.set_ylabel("COR3M − realized next 63d (points)")
    a1.set_title("Correlation risk premium, ex post", loc="left")
    a1.legend(loc="lower left", fontsize=8.5)
    a2.hist(prem, bins=40, color=C["blue"], edgecolor=C["surface"], linewidth=0.6, orientation="horizontal")
    a2.axhline(prem.median(), color=C["ink"], lw=1.2, ls="--")
    a2.text(a2.get_xlim()[1] * 0.98, prem.median(), f"median {prem.median():.1f}", ha="right", va="bottom", fontsize=8.5, color=C["ink"])
    a2.set_xlabel("Days"); a2.set_title("Distribution", loc="left")
    a2.set_ylim(a1.get_ylim())
    fig.tight_layout()
    save_fig(fig, "02_correlation_risk_premium.png")
    PREM = {"mean": float(prem.mean()), "median": float(prem.median()), "pct_pos": float((prem > 0).mean()),
            "p5": float(prem.quantile(0.05)), "p95": float(prem.quantile(0.95)),
            "start": str(prem.index[0].date()), "end": str(prem.index[-1].date())}
    print(PREM)

# --- Figure 3: backtest P&L and drawdown ----------------------------------------------------------
if BACKTEST_OK:
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 5.6), sharex=True, gridspec_kw={"height_ratios": [2.2, 1]})
    shade_stress(a1, vix.loc[pnl_mtm.index[0]:] if vix is not None else None, CFG["STRESS_VIX"])
    a1.plot(pnl_mtm, color=C["blue"], lw=1.8, label="Mark-to-market (approx.)")
    a1.plot(pnl_hold, color=C["ink2"], lw=1.0, ls="--", label="Booked at expiry only")
    a1.axhline(0, color=C["ink2"], lw=0.6)
    a1.set_ylabel("Cum. P&L (corr. points)")
    a1.set_title("Short 3M correlation struck at COR3M, daily ladder — no costs", loc="left")
    a1.legend(loc="upper left", fontsize=8.5)
    a2.fill_between(dd.index, dd, 0, color=C["red"], lw=0)
    a2.set_ylabel("Drawdown (pts)")
    a2.annotate(f"max DD {BT['max_dd_pts']:.1f} pts\n{BT['dd_peak']} → {BT['dd_trough']}",
                xy=(trough, dd.min()), xytext=(10, 6), textcoords="offset points", fontsize=8.5, color=C["ink"])
    fig.tight_layout()
    save_fig(fig, "03_backtest_pnl_drawdown.png")

# --- Figure 4: today's vol snapshot (3M) ----------------------------------------------------------
d3 = comp[(comp["tenor"] == "3M") & comp["vol"].between(0.03, 3.0)].sort_values("w", ascending=False).head(25)
fig, ax = plt.subplots(figsize=(11, 4.6))
ax.bar(d3["ticker"], d3["vol"] * 100, color=C["orange"], width=0.65, label="Component 3M ATM implied vol")
ax.axhline(snap["3M"]["wavg_comp_vol"] * 100, color=C["ink"], ls="--", lw=1.2,
           label=f"Weighted avg. component vol (top {snap['3M']['n_names']}): {snap['3M']['wavg_comp_vol']*100:.1f}")
ax.axhline(snap["3M"]["index_vol"] * 100, color=C["blue"], lw=2.0,
           label=f"{INDEX_OPT_SYMBOL} 3M ATM implied vol: {snap['3M']['index_vol']*100:.1f}")
ax.set_ylabel("Implied vol (%)"); ax.tick_params(axis="x", rotation=60, labelsize=8); ax.grid(axis="x", visible=False)
ttl = f"3M ATM vols on {VAL_DATE.date()} — implied correlation {snap['3M']['implied_corr']*100:.1f}"
if K.notna().any():
    ttl += f" (Cboe COR3M: {K.dropna().iloc[-1]*100:.1f})"
ax.set_title(ttl, loc="left")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=3, fontsize=8.5)
ax.text(0, -0.40, "25 largest weights shown; ATM-forward vols from Yahoo option mids, parity-implied forwards, "
        "total-variance interpolation to 91 days.", transform=ax.transAxes, fontsize=8, color=C["ink2"])
save_fig(fig, "04_vol_snapshot.png")

## 7. Current levels, signal, carry and sizing → `levels.json` and README auto-fill

**Carry.** If realized correlation over the next 3M equals the trailing 3M realized, a short-correlation
position earns roughly *COR3M − realized 3M* points. In vol terms: with today's implied single-stock vols and
trailing realized correlation, the "fair" index vol would be
$\sigma_I^{fair}=\sqrt{\sum w_i^2\sigma_i^2+\rho^{real}\,[(\sum w_i\sigma_i)^2-\sum w_i^2\sigma_i^2]}$.

**Theta-neutral sizing.** Allocate component vega $v_i = w_i\,v_I\,\sigma_I/\bar\sigma$ with
$\bar\sigma=\sum w_i\sigma_i$. Then $\sum v_i\sigma_i = v_I\sigma_I$ (theta-neutral for ATM straddles, since
$\Theta\approx-\text{vega}\cdot\sigma/2T$) and the book is flat to **proportional** vol moves, which leave implied
correlation unchanged. Its first-order exposure is to correlation:
$\partial\sigma_I/\partial\rho = \frac{(\sum w_i\sigma_i)^2-\sum w_i^2\sigma_i^2}{2\sigma_I}$.
**Vega-neutral** ($v_i=w_i v_I$) is flat to equal vol-point moves but pays more theta and is long vol.

In [ ]:
def fnum(x, nd=1, pct=False):
    if x is None or not np.isfinite(x): return "n/a"
    return f"{x*100:.{nd}f}%" if pct else f"{x:.{nd}f}"

d = comp[(comp["tenor"] == "3M") & comp["vol"].between(0.03, 3.0)]
wv = (d["w"] / d["w"].sum()).values; sv = d["vol"].values
# Sizing for the book actually traded: top TRADE_N names, weights renormalized inside that basket
dt = d.sort_values("w", ascending=False).head(CFG["TRADE_N"])
wt = (dt["w"] / dt["w"].sum()).values; st_ = dt["vol"].values
at_, bt_ = np.sum(wt**2 * st_**2), np.sum(wt * st_)**2
TRADE_IDX_WEIGHT = float(top.set_index("Ticker").loc[dt["ticker"], "Weight"].sum() / holdings["Weight"].sum())
a_ = np.sum(wv**2 * sv**2); b_ = np.sum(wv * sv)**2
sig_I = snap["3M"]["index_vol"]; sbar = np.sum(wv * sv)
rho_real3 = float(rc["3M"].dropna().iloc[-1]); rho_real1 = float(rc["1M"].dropna().iloc[-1])
sig_fair = np.sqrt(a_ + rho_real3 * (b_ - a_))
dsig_drho = (bt_ - at_) / (2 * sig_I)                       # decimal vol per unit correlation, traded basket

L = {"VAL_DATE": str(VAL_DATE.date()), "WEIGHT_SOURCE": WEIGHT_SOURCE, "N": str(len(W)),
     "INDEX_COVERAGE": fnum(INDEX_COVERAGE, 0, True), "INDEX_OPT": INDEX_OPT_SYMBOL,
     "IDX_VOL_1M": fnum(snap["1M"]["index_vol"], 1, True), "IDX_VOL_3M": fnum(sig_I, 1, True),
     "COMP_VOL_3M": fnum(sbar, 1, True), "COMP_VOL_1M": fnum(snap["1M"]["wavg_comp_vol"], 1, True),
     "IC_1M": fnum(snap["1M"]["implied_corr"] * 100), "IC_3M": fnum(snap["3M"]["implied_corr"] * 100),
     "IC_3M_PROXY": fnum(snap["3M"]["implied_corr_proxy"] * 100),
     "NAMES_3M": str(snap["3M"]["n_names"]), "LASTPX_SHARE": fnum(snap["3M"]["share_last_price"], 0, True),
     "RC_1M": fnum(rho_real1 * 100), "RC_3M": fnum(rho_real3 * 100), "RC_3M_BASKET": fnum(BASKET_PAIRWISE_3M * 100),
     "VIX": fnum(cboe["^VIX"].dropna().iloc[-1] * 100) if "^VIX" in cboe and cboe["^VIX"].notna().any() else "n/a",
     "VIX3M": fnum(cboe["^VIX3M"].dropna().iloc[-1] * 100) if "^VIX3M" in cboe and cboe["^VIX3M"].notna().any() else "n/a",
     "SIG_FAIR": fnum(sig_fair, 1, True), "IDX_RICH_VOLPTS": fnum((sig_I - sig_fair) * 100),
     "VEGA_RATIO": fnum(sig_I / np.sum(wt * st_), 2), "CORR_VEGA_100K": f"{100_000 * dsig_drho * 100 * 0.01:,.0f}",
     "TRADE_N": str(len(dt)), "TRADE_IDX_WEIGHT": fnum(TRADE_IDX_WEIGHT, 0, True)}
# CORR_VEGA_100K: $ P&L per 1 correlation point, per $100k of index vega (index vega in $ per vol point)

for tk_, key in (("^COR1M", "COR1M"), ("^COR3M", "COR3M")):
    s_ = cboe[tk_].dropna() if tk_ in cboe else pd.Series(dtype=float)
    L[key] = fnum(s_.iloc[-1] * 100) if len(s_) else "n/a"
    L[key + "_DATE"] = str(s_.index[-1].date()) if len(s_) else "n/a"
    L[key + "_PCTL"] = fnum((s_ <= s_.iloc[-1]).mean(), 0, True) if len(s_) else "n/a"
    L[key + "_SINCE"] = str(s_.index[0].date()) if len(s_) else "n/a"

# Sanity check of our snapshot against Cboe (different vol source/timing => a few points of gap is normal)
gap = snap["3M"]["implied_corr"] * 100 - (K.dropna().iloc[-1] * 100 if K.notna().any() else np.nan)
L["SNAPSHOT_CHECK"] = ("n/a (no COR3M)" if not np.isfinite(gap) else
    f"{gap:+.1f} pts vs COR3M" + (" — LARGE GAP: check quotes before using the snapshot" if abs(gap) > 10 else ""))
print("Snapshot vs COR3M:", L["SNAPSHOT_CHECK"])

SIGNAL = "n/a"
if BACKTEST_OK:
    cor_now = float(K.dropna().iloc[-1])
    spread_hist = ((K - rc["3M"]) * 100).dropna()          # implied minus TRAILING realized (known at t)
    spread_now = (cor_now - rho_real3) * 100
    ts_ratio = float(cboe["^VIX"].dropna().iloc[-1] / cboe["^VIX3M"].dropna().iloc[-1]) if "^VIX3M" in cboe and cboe["^VIX3M"].notna().any() else np.nan
    chg21 = (K * 100).diff(21).dropna()
    stop_level = cor_now * 100 + chg21.quantile(0.95)
    THESIS = ("Index options currently price the largest S&P 500 stocks to move together more than they have "
              "recently done." if spread_now > 0 else
              "Today, index options do NOT price more co-movement than recently realized: the structural premium "
              "is absent, so this pitch is a watch-list idea with an entry level, not a live trade.")
    if spread_now <= 0:
        SIGNAL = "NO TRADE — implied correlation is not above trailing realized"
    elif spread_now > spread_hist.median() and (not np.isfinite(ts_ratio) or ts_ratio < 1):
        SIGNAL = "ENTER — implied-realized spread above its historical median, VIX curve in contango"
    else:
        SIGNAL = "WATCH — premium positive but below median and/or VIX curve inverted"
    L.update({"SPREAD_NOW": fnum(spread_now), "SPREAD_MEDIAN": fnum(spread_hist.median()),
              "SPREAD_PCTL": fnum((spread_hist <= spread_now).mean(), 0, True),
              "VIX_TS": fnum(ts_ratio, 2), "STOP_COR3M": fnum(stop_level),
              "CHG21_P95": fnum(chg21.quantile(0.95)), "SIGNAL": SIGNAL, "THESIS": THESIS,
              "BT_DROPPED": str(BT["n_dropped"]),
              "HAIRCUT_TABLE": "| Strike vs COR3M | P&L (pts/yr) | Sharpe | Max DD (pts) | Hit rate |\n|---|---|---|---|---|\n" + "\n".join(
                  f"| COR3M − {r['haircut_pts']:.0f} pts | {r['ann_pts']:.1f} | {r['sharpe']:.2f} | {r['max_dd_pts']:.1f} | {r['hit_rate']*100:.0f}% |"
                  for _, r in haircut_df.iterrows()),
              "BT_START": BT["start"], "BT_END": BT["end"], "BT_YEARS": fnum(BT["years"]),
              "BT_ANN": fnum(BT["ann_pts"]), "BT_VOL": fnum(BT["ann_vol_pts"]), "BT_SHARPE": fnum(BT["sharpe"], 2),
              "BT_MAXDD": fnum(BT["max_dd_pts"]), "BT_DD_PEAK": BT["dd_peak"], "BT_DD_TROUGH": BT["dd_trough"],
              "BT_WORST_DAY": fnum(BT["worst_day_pts"]), "BT_WORST_DAY_DATE": BT["worst_day"],
              "BT_HIT": fnum(BT["hit_rate"], 0, True), "BT_AVG_TRANCHE": fnum(BT["avg_tranche_pts"]),
              "BT_P5_TRANCHE": fnum(BT["p5_tranche_pts"]),
              "PREM_MEDIAN": fnum(PREM["median"]), "PREM_POS": fnum(PREM["pct_pos"], 0, True), "PREM_P5": fnum(PREM["p5"])})
    st_md = "| Window | P&L (pts) | COR3M start → max | Realized 1M max |\n|---|---|---|---|\n" + "\n".join(
        f"| {r['window']} | {r['pnl_pts']:+.1f} | {r['COR3M_start']:.1f} → {r['COR3M_max']:.1f} | {r['realized1M_max']:.1f} |"
        for _, r in stress_df.iterrows()) if len(stress_df) else "_No named stress window inside the COR3M sample._"
    L["STRESS_TABLE"] = st_md

with open(os.path.join(CFG["OUT"], "levels.json"), "w") as f:
    json.dump(L, f, indent=1)
print(json.dumps(L, indent=1))

# Auto-fill README_template.md -> output/README.md (local file first, then the GitHub raw URL)
tpl = None
for pth in ("README_template.md", "trade-ideas/01-dispersion/README_template.md"):
    if os.path.exists(pth):
        tpl = open(pth, encoding="utf-8").read(); break
if tpl is None and CFG["README_TEMPLATE_URL"]:
    try:
        r = requests.get(CFG["README_TEMPLATE_URL"], timeout=20)
        if r.ok: tpl = r.text
    except Exception as e:
        print("template download failed:", e)
if tpl:
    out = tpl
    for k_, v_ in L.items():
        out = out.replace("{{" + k_ + "}}", str(v_))
    left = sorted(set(re.findall(r"\{\{(\w+)\}\}", out)))
    with open(os.path.join(CFG["OUT"], "README.md"), "w", encoding="utf-8") as f:
        f.write(out)
    print("README written to output/README.md", "| unfilled placeholders:", left or "none")
else:
    print("No README template found: upload README_template.md to the Colab session to auto-fill the pitch.")

## 8. Limitations (read before quoting any number)
* **Snapshot implied correlation** depends on Yahoo quote quality; outside US hours quotes may be stale
  (`share_last_price` reports the basket weight priced on last trades). Compare with COR3M: a gap of a few points
  is normal (different vol source, ATM definition, timing); a large gap means bad quotes.
* **Look-ahead / survivorship**: today's top-N names and weights are applied to the past; historically the top 50
  were different (e.g. more financials and energy), which biases realized correlation.
* **Backtest = correlation swap proxy**, not a straddle dispersion: no vega/vol-level P&L, no gamma path
  dependency, no single-stock jumps (earnings, M&A), no transaction costs, no margin. Bid/ask on 50 single-stock
  straddles is a large cost in practice; real books trade the top 20–30 names.
* **MtM approximation** uses COR3M at t as the expectation of remaining correlation (tenor mismatch as the
  tranche ages) and adds sub-period correlations linearly.
* COR3M history length is whatever Yahoo serves; the sample may not contain every historical crisis.

In [ ]:
# Zip everything for download (Colab: Files panel -> output.zip)
import shutil
shutil.make_archive("output", "zip", CFG["OUT"])
print("output.zip ready")
if IN_COLAB:
    from google.colab import files  # noqa
    files.download("output.zip")